# investalyze: motifs & transitions

The pipeline payload: cluster history vectors into recurring **motifs**, then study the **futures** that followed each motif. Builds directly on notebooks 3 (segments + encodings) and 4 (which validates that similar vectors really are similar).

> **Status: skeleton.** The roadmap below lists what to build. Implement top to bottom.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
from helpers import connect_readonly

from investalyze.analysis import encodings, segments

con = connect_readonly()

## 1. Build the vectors (shared preamble)

Same preamble as notebook 4: `load_series` → `build_segments` → split at `HISTORY_LENGTH` → `encodings.rebase_to_100(history)` to get `X`. Keep `future` and `meta` aligned by row position, `meta.loc[i]` describes `X[i]` / `future[i]`.

_If this preamble starts being copy-pasted across notebooks 4 and 5, lift it into a helper in `notebooks/helpers.py` (e.g. `load_vectors(con, ...)` returning `X, future, meta`)._

In [ ]:
# TODO: paste the notebook-4 preamble (CLASSES/TICKERS, lengths, load+build+split+encode -> X, future, meta)

## 2. Cluster histories into motifs

**Goal:** group `X` into a handful of recurring shapes.

- No sklearn in the env yet, either add `scikit-learn` (KMeans / AgglomerativeClustering) or use `scipy.cluster` (`scipy.cluster.vq.kmeans2`, or `scipy.cluster.hierarchy` for agglomerative). Decide and note the choice.
- Pick `k` deliberately: elbow on inertia, or silhouette. Start small (k≈6-10).
- Attach the label to `meta` (`meta["motif"] = labels`) so futures can be grouped by it.
- **Plot each motif:** its centroid plus a sample of member histories (rebased), one panel per motif, with the member count in the title.

In [ ]:
# TODO: fit clustering on X -> labels; meta["motif"] = labels; plot centroid + members per motif

## 3. Transition analysis (futures per motif)

**Goal:** for each motif, characterise what came next.

- For motif `m`, take `future[meta["motif"] == m]`. Encode the futures **continuously with their histories** so paths are comparable, rebase each full window to 100 and slice off the future part (or reuse `rebase_to_100(history, future)`, which rebases the future on the history's base).
- **Mean future path** + dispersion band (e.g. 25-75th percentile) per motif, overlaid on the motif centroid so the join is visible at the history→future seam.
- **Up/down odds:** share of futures ending above their last history price; mean / median forward return; dispersion. Tabulate per motif.
- Optional **Markov view:** if you also label the *future* window with a motif, build the motif→next-motif transition matrix.

In [ ]:
# TODO: per-motif futures -> mean path + dispersion band; up/down odds table; optional transition matrix

## Open decisions

- **Encoding for clustering:** rebase-to-100 (amplitude-aware) vs z-score (pure shape). Notebook 3 leans rebase-to-100 for motifs; revisit if clusters are dominated by volatility.
- **Clustering library:** add scikit-learn, or stay scipy-only.
- **k selection:** elbow vs silhouette vs fixed small k.
- **Overlap:** STRIDE=20 == HISTORY_LENGTH gives non-overlapping histories (independent samples). Smaller stride = more, correlated windows, fine for shape mining, but note it.